# AO3 Fandom Category Atlas: 抓取工作流

这个 notebook 是项目 **AO3 Fandom Category Atlas** 的抓取入口，用来运行并检查 [extras/ao3_comments_tags_spider.py](extras/ao3_comments_tags_spider.py)。

流程：
1. 从 `ao3.xlsx` 读取 `source_label` 和 `start_url` 任务。
2. 配置抓取页数、抓取上限和可选 cookie。
3. 通过 `python -m scrapy runspider` 启动 spider。
4. 读取导出的 JSONL，快速检查作品和评论是否抓到。
5. 把作品数据展开为 Excel，供后续分类分析 notebook 使用。

In [1]:
from __future__ import annotations

import json
import pandas as pd
import subprocess
import sys
from collections import Counter
from pathlib import Path

ROOT = Path.cwd()
PROJECT_ROOT = ROOT.parent if ROOT.name == "src" else ROOT
SPIDER_FILE = ROOT / "extras" / "ao3_comments_tags_spider.py"
INPUT_EXCEL_PATH = PROJECT_ROOT / "ao3.xlsx"
OUTPUT_DIR = ROOT / "output"
RUN_OUTPUT_PATH = OUTPUT_DIR / "_ao3_current_run.jsonl"
OUTPUT_PATH = OUTPUT_DIR / "ao3_jujutsu_comments_tags.jsonl"
EXCEL_OUTPUT_PATH = OUTPUT_DIR / "ao3_jujutsu_comments_tags.xlsx"

print("ROOT:", ROOT)
print("SPIDER_FILE exists:", SPIDER_FILE.exists())
print("INPUT_EXCEL_PATH exists:", INPUT_EXCEL_PATH.exists())

ROOT: f:\Desktop\scrapy-master\src
SPIDER_FILE exists: True


In [2]:
MAX_PAGES = 1
MAX_WORKS = 3
COOKIE_HEADER = ""

# 如果 AO3 需要登录或要访问成人内容，把浏览器里的 cookie 粘到 COOKIE_HEADER。
# 例如：
# COOKIE_HEADER = "_otwarchive_session=...; user_credentials=..."

OVERWRITE_OUTPUT = True
LOG_LEVEL = "INFO"

input_df = pd.read_excel(INPUT_EXCEL_PATH, header=None)
input_df = input_df.iloc[:, :2].copy()
input_df.columns = ["source_label", "start_url"]
input_df = input_df.dropna(subset=["source_label", "start_url"])
crawl_tasks = input_df.to_dict("records")

print("Crawl tasks:", len(crawl_tasks))
print(input_df.head())

In [3]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not SPIDER_FILE.exists():
    raise FileNotFoundError(f"Spider file not found: {SPIDER_FILE}")
if not INPUT_EXCEL_PATH.exists():
    raise FileNotFoundError(f"Input Excel not found: {INPUT_EXCEL_PATH}")

all_items = []
run_summaries = []

for idx, task in enumerate(crawl_tasks, start=1):
    source_label = str(task["source_label"]).strip()
    start_url = str(task["start_url"]).strip()
    if not source_label or not start_url:
        continue

    if RUN_OUTPUT_PATH.exists():
        RUN_OUTPUT_PATH.unlink()

    cmd = [
        sys.executable,
        "-m",
        "scrapy",
        "runspider",
        str(SPIDER_FILE),
        "-a",
        f"source_label={source_label}",
        "-a",
        f"start_url={start_url}",
        "-a",
        f"max_pages={MAX_PAGES}",
        "-a",
        f"max_works={MAX_WORKS}",
        "-s",
        f"LOG_LEVEL={LOG_LEVEL}",
    ]

    if COOKIE_HEADER.strip():
        cmd.extend(["-a", f"cookie_header={COOKIE_HEADER}"])

    cmd.extend([
        "-O" if OVERWRITE_OUTPUT else "-o",
        str(RUN_OUTPUT_PATH),
    ])

    print(f"\n[{idx}/{len(crawl_tasks)}] Running command for {source_label}:")
    print(" ".join(cmd))

    process = subprocess.Popen(
        cmd,
        cwd=ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )

    captured_lines = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        captured_lines.append(line)

    result_code = process.wait()
    print("\nReturn code:", result_code)

    if result_code != 0:
        print("\nLast output lines:\n")
        print("".join(captured_lines[-40:]) if captured_lines else "<empty>")
        raise RuntimeError(f"Spider run failed for {source_label}.")

    if not RUN_OUTPUT_PATH.exists():
        raise FileNotFoundError(f"Output file was not created: {RUN_OUTPUT_PATH}")

    run_items = []
    with RUN_OUTPUT_PATH.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                run_items.append(json.loads(line))

    all_items.extend(run_items)
    run_summaries.append({"source_label": source_label, "start_url": start_url, "item_count": len(run_items)})
    print(f"Items scraped for {source_label}: {len(run_items)}")

with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    for item in all_items:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"\nCombined output written to: {OUTPUT_PATH}")
print(pd.DataFrame(run_summaries).head())

Running command:
d:\AnacondaData\envs_dirs\Scraper\python.exe -m scrapy runspider f:\Desktop\scrapy-master\src\extras\ao3_comments_tags_spider.py -a start_url=https://archiveofourown.org/works?commit=Sort+and+Filter&work_search[sort_column]=hits&work_search[other_tag_names]=&work_search[excluded_tag_names]=&work_search[crossover]=&work_search[complete]=&work_search[words_from]=&work_search[words_to]=&work_search[date_from]=&work_search[date_to]=&work_search[query]=&work_search[language_id]=&tag_id=%E5%91%AA%E8%A1%93%E5%BB%BB%E6%88%A6+%7C+Jujutsu+Kaisen+%28Anime+*a*+Manga%29&page=1 -a max_pages=1 -a max_works=3 -s LOG_LEVEL=INFO -O f:\Desktop\scrapy-master\src\output\ao3_jujutsu_comments_tags.jsonl
2026-03-10 23:02:07 [scrapy.utils.log] INFO: Scrapy 2.14.1 started (bot: scrapybot)
2026-03-10 23:02:07 [scrapy.utils.log] INFO: Versions:
{'lxml': '6.0.2',
 'libxml2': '2.11.9',
 'cssselect': '1.4.0',
 'parsel': '1.11.0',
 'w3lib': '2.4.0',
 'Twisted': '25.5.0',
 'Python': '3.10.19 | package

In [4]:
items = []
with OUTPUT_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            items.append(json.loads(line))

print("Total items:", len(items))
print("By type:", Counter(item.get("item_type", "unknown") for item in items))

works = [item for item in items if item.get("item_type") == "work"]
comments = [item for item in items if item.get("item_type") == "comment"]

print("Works:", len(works))
print("Comments:", len(comments))

Total items: 3
By type: Counter({'work': 3})
Works: 3
Comments: 0


In [5]:
def _join_excel_value(value):
    if value is None:
        return None
    if isinstance(value, list):
        return " | ".join(str(v) for v in value if v not in (None, "")) or None
    if isinstance(value, dict):
        return json.dumps(value, ensure_ascii=False)
    return value

def flatten_work_for_excel(work):
    row = {
        "source_label": work.get("source_label"),
        "work_id": work.get("work_id"),
        "title": work.get("title"),
        "authors": _join_excel_value(work.get("authors")),
        "work_url": work.get("work_url"),
        "search_page": work.get("search_page"),
        "published_or_updated": work.get("published_or_updated"),
        "summary": work.get("summary"),
        "fandoms": _join_excel_value(work.get("fandoms")),
        "series": _join_excel_value(work.get("series")),
        "gift_recipients": _join_excel_value(work.get("gift_recipients")),
        "language": work.get("language"),
        "words": work.get("words"),
        "chapters": work.get("chapters"),
        "collections": work.get("collections"),
        "comments_total": work.get("comments_total"),
        "kudos": work.get("kudos"),
        "bookmarks": work.get("bookmarks"),
        "hits": work.get("hits"),
    }

    for key, value in (work.get("icon_info") or {}).items():
        row[f"icon_{key}"] = _join_excel_value(value)

    for group_name, values in (work.get("listing_tags") or {}).items():
        row[f"tag_{group_name}"] = _join_excel_value(values)

    return row

work_rows = [flatten_work_for_excel(work) for work in works]
df = pd.DataFrame(work_rows)
df.to_excel(EXCEL_OUTPUT_PATH, index=False)

print(f"Excel written to: {EXCEL_OUTPUT_PATH}")
print(df.head())

Excel written to: f:\Desktop\scrapy-master\src\output\ao3_jujutsu_comments_tags.xlsx
    work_id                                              title    authors  \
0  39734967                                    zenith of stars     Yuesya   
1  35559262  helping the world via murder, a guide by your ...     RK7200   
2  54460024                                  To Tame A Monster  tonycries   

                                     work_url  search_page  \
0  https://archiveofourown.org/works/39734967            1   
1  https://archiveofourown.org/works/35559262            1   
2  https://archiveofourown.org/works/54460024            1   

  published_or_updated                                            summary  \
0          07 Mar 2026  "It's not the Six Eyes," they said. So then, t...   
1          05 Jan 2026  Cursed spirits are usually born knowing what t...   
2          05 Mar 2026  “Just a small initiation, nothing too serious....   

                                             fan

In [6]:
sample_work = works[0] if works else None
sample_comment = comments[0] if comments else None

print("Sample work:\n")
print(json.dumps(sample_work, ensure_ascii=False, indent=2) if sample_work else "<no work items>")

print("\nSample comment:\n")
print(json.dumps(sample_comment, ensure_ascii=False, indent=2) if sample_comment else "<no comment items>")

Sample work:

{
  "item_type": "work",
  "work_id": "39734967",
  "search_page": 1,
  "work_url": "https://archiveofourown.org/works/39734967",
  "title": "zenith of stars",
  "authors": [
    "Yuesya"
  ],
  "fandoms": [
    "呪術廻戦 | Jujutsu Kaisen (Manga)",
    "呪術廻戦 | Jujutsu Kaisen (Anime)"
  ],
  "summary": "\"It's not the Six Eyes,\" they said. So then, the question is –what is it? Alternatively: A young girl with cursed eyes finds her way forward in a world of sorcery and curses. [OC, Mystic Eyes of Death Perception!OC, AU]",
  "series": [],
  "gift_recipients": [],
  "icon_info": {
    "rating_code": "rating-teen",
    "rating_text": "Teen And Up Audiences",
    "warning-choosenotto": "Creator Chose Not To Use Archive Warnings",
    "category_code": "category-none",
    "category_text": "No category",
    "complete_code": "complete-no",
    "complete_text": "Work in Progress"
  },
  "listing_tags": {
    "ratings": [
      "Teen And Up Audiences"
    ],
    "categories": [
     